# 34B The query server
Four READ tools, a closed catalog, a read-only connection, and an audit event for every call. The agent reaches data through nothing else.

In [ ]:
import sys, json, pathlib, tempfile
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "askpanel").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, yaml
from askpanel import audit, catalog, clock, db, econ, gate, llm, nodes, questions, standin, tools
P = yaml.safe_load(open(ROOT / "params.yaml"))
PANEL = db.load_panel(ROOT / P["data"]["panel"])
TMP = pathlib.Path(tempfile.mkdtemp())
BASE = db.build_db(PANEL, TMP / "base.sqlite")          # tables only: works before you write the view
print(f"{len(PANEL)} panel rows, {PANEL['adsh'].nunique()} filings; scratch database {BASE.name}")

In [ ]:
led = audit.Ledger(clock=clock.clock)
T = tools.local_tools(BASE, led, tables=("filings", "holdings"))
print(T["list_tables"]())
print([c["column"] for c in T["describe_table"](table="filings")])
print(T["sample_values"](table="filings", column="name"))
print(T["run_query"](sql="SELECT COUNT(*) AS n FROM holdings")["rows"])
try:
    T["run_query"](sql="DROP TABLE holdings")
except gate.GateError as e:
    print("refused:", e.code)
for e in led.events[-2:]:
    print(e)

In [ ]:
mine = dict(catalog.CATALOG); mine.pop("sample_values"); mine["export_panel"] = "READ"
print(catalog.conformance(mine))
print(catalog.conformance(dict(catalog.CATALOG)), "<- what your REGISTRY must give")

## Wrapping it in MCP (SDK v2)
```python
from mcp.server import MCPServer            # pip install "mcp[cli]>=2.1,<3"
mcp = MCPServer("askpanel-query")

@mcp.tool()
def run_query(sql: str) -> QueryResult:     # a Pydantic model: it becomes structured_content
    """Run ONE SELECT (READ). Refused unless read-only; capped rows; timed out."""
    return QueryResult(**gate.run_query(con(), sql))
```
The cell below calls a tool over the protocol, in memory, if `mcp` is installed.

In [ ]:
import importlib.util, asyncio, concurrent.futures
from typing import Any
if importlib.util.find_spec("mcp") is None:
    print("mcp not installed: skipped (pip install 'mcp[cli]>=2.1,<3')")
else:
    try:
        from mcp.server import MCPServer
        from mcp.server.mcpserver.exceptions import ToolError
        from mcp import Client
        srv = MCPServer("askpanel-demo")

        @srv.tool()
        def run_query(sql: str) -> dict[str, Any]:   # v2: a bare dict gives structured_content None
            """Run ONE SELECT (READ)."""
            try:
                return T["run_query"](sql=sql)
            except gate.GateError as e:
                raise ToolError(str(e)) from None   # v2: only ToolError keeps its message (and is not logged as a crash)

        async def demo():
            async with Client(srv) as c:                       # in memory: no transport
                ok = await c.call_tool("run_query", {"sql": "SELECT COUNT(*) AS n FROM holdings"})
                bad = await c.call_tool("run_query", {"sql": "DROP TABLE holdings"})
                return ok.structured_content, bad.is_error
        # Jupyter already runs an event loop, so asyncio.run() is refused there: run demo() in its own thread
        with concurrent.futures.ThreadPoolExecutor(1) as pool:
            print(pool.submit(asyncio.run, demo()).result())
    except Exception as e:
        print("MCP demo failed:", type(e).__name__, e, "(check the SDK version against the v2 docs)")

**Your turn (Part 2, 18 points).**
1. `services/mcp/server.py`: the four tools, Pydantic output models, `REGISTRY = {...}` as a literal, the database opened with `gate.connect_ro`, an audit event per call and per refusal.
2. `tests/test_server.py`: over `Client(server.mcp)`, the catalog is closed, a SELECT works, and `INSERT`, `DROP`, `ATTACH`, `PRAGMA`, `sqlite_master` and a `;`-joined second statement are all refused.
3. `mcp dev services/mcp/server.py`: call each tool from the Inspector once (a screenshot goes in your README).